# Step-1 : Install  the required libraries/packages

In [ ]:
#pip install pandas
#pip install openpyxl
#pip install duckdb


# Step-2 : Import the required libraries/packages

In [ ]:
import pandas as pd
import sqlite3
import duckdb
import os 

# Step-3 : Left join
## Datasets using : 
- Product_sample_500
- Links

In [ ]:
'''-----Left join-----'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)

# Create SQL memory DB
conn = sqlite3.connect(":memory:")

# Load tables
product_df.to_sql("product_sample_500", conn, index=False, if_exists="replace")
links_df.to_sql("links", conn, index=False, if_exists="replace")

# Overlay logic
query = """
SELECT 
    p.*,
    COALESCE(p.GMID_CD, l.GMID_CD) AS FILLED_GMID
FROM product_sample_500 p
LEFT JOIN links l
ON p.product_id = l.product_id
"""

# Execute query
final_product = pd.read_sql_query(query, conn)

# Replace original GMID
final_product["GMID_CD"] = final_product["FILLED_GMID"]
final_product.drop(columns=["FILLED_GMID"], inplace=True)

# Create output folder
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# Save location
file_path = os.path.join(output_folder, "product_sample_500_cleaned.xlsx")

# Save result file with sheet name
final_product.to_excel(file_path, sheet_name="Left_Join", index=False)

print("Result file saved in:", file_path)

Result file saved in: output\product_sample_500_cleaned.xlsx


# Step-4 : Right join
## Datasets using : 
- Product_sample_500
- Links

In [14]:
'''------Right join------'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)
links_df = links_df.replace('', pd.NA)

# Connect to DuckDB
conn = duckdb.connect()

# Register dataframes as tables
conn.register("product_sample_500", product_df)
conn.register("links", links_df)

# RIGHT JOIN query
query = """
SELECT 
    l.*,
    COALESCE(l.GMID_CD, p.GMID_CD) AS FILLED_GMID
FROM product_sample_500 p
RIGHT JOIN links l
ON p.product_id = l.product_id
"""

# Execute query
final_links = conn.execute(query).df()

# Replace GMID column
final_links["GMID_CD"] = final_links["FILLED_GMID"]
final_links.drop(columns=["FILLED_GMID"], inplace=True)

# Output folder
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# Excel file path
file_path = os.path.join(output_folder, "product_sample_500_cleaned.xlsx")

# Write to Excel
if os.path.exists(file_path):

    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        final_links.to_excel(writer, sheet_name="Right_Join", index=False)

else:

    with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
        final_links.to_excel(writer, sheet_name="Right_Join", index=False)

print("Right Join result saved in:", file_path)

Right Join result saved in: output\product_sample_500_cleaned.xlsx


# Step-5 : Full join
## Datasets using : 
- Product_sample_500
- Links

In [15]:
'''-----Full join-----'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)
links_df = links_df.replace('', pd.NA)

# Connect DuckDB
conn = duckdb.connect()

# Register tables
conn.register("product", product_df)
conn.register("links", links_df)

# FULL OUTER JOIN query
query = """
SELECT *
FROM product p
FULL OUTER JOIN links l
ON p.product_id = l.product_id
"""

# Execute query
full_join = conn.execute(query).df()

# Output folder
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

# Excel file path
file_path = os.path.join(output_folder, "product_sample_500_cleaned.xlsx")

# Write to Excel
if os.path.exists(file_path):

    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        full_join.to_excel(writer, sheet_name="Full_Join", index=False)

else:

    with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
        full_join.to_excel(writer, sheet_name="Full_Join", index=False)

print("Full Join result saved in:", file_path)

Full Join result saved in: output\product_sample_500_cleaned.xlsx
